In [1]:
import pandas as pd
import numpy as np

In [2]:
url="https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"

In [3]:
column_names = [
    "Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight",
    "Viscera_weight", "Shell_weight", "Rings"
]

df = pd.read_csv(url, header=None, names=column_names)

In [4]:
df.shape

(4177, 9)

In [5]:
df.columns.tolist()

['Sex',
 'Length',
 'Diameter',
 'Height',
 'Whole_weight',
 'Shucked_weight',
 'Viscera_weight',
 'Shell_weight',
 'Rings']

In [6]:
df.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [ ]:
# input= numeric measurements like length, diameter, height, weights, etc
# output= age of abalone
# output is numeric because age=rings + 1.5 which is continuous numeric value

In [7]:
df["y"] = df["Rings"] + 1.5
df.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings,y
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,16.5
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,8.5
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,10.5
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,11.5
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,8.5


In [12]:
features = ["Length", "Diameter", "Whole_weight"]
X = df[features].values   
y = df["y"].values    
# Length and Diameter represent physical growth.
# Whole_weight correlates with maturity and age.
# All are continuous numeric measurements suitable for linear regression.

In [13]:
# Train-test split (80/20)

np.random.seed(42)
indices = np.random.permutation(len(X))
split_idx = int(0.8 * len(X))
train_idx = indices[:split_idx]
test_idx = indices[split_idx:]
X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(3341, 3)
(836, 3)
(3341,)
(836,)


In [14]:
# Normalize inputs

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

print(mean)
print(std)

[0.52164771 0.40591589 0.81972359]
[0.12099972 0.09981478 0.48978825]


In [ ]:
# why normalization is needed for learning:
# features have different scales (weight vs length)
# leads to faster and more stable convergence

In [18]:
d = X_train.shape[1]
w = np.zeros((d, 1))   
b = 0.0                

print("Initialized w shape:", w.shape)
print("Initialized b type :", type(b))

def forward(X, w, b):
    y_hat = X @ w + b   
    print(X.shape)
    print(w.shape)
    print(np.isscalar(b))
    print(y_hat.shape)
    return y_hat

# Reshape y for consistency
y_train_reshaped = y_train.reshape(-1, 1)

# forward pass
y_hat = forward(X_train, w, b)

Initialized w shape: (3, 1)
Initialized b type : <class 'float'>
(3341, 3)
(3, 1)
True
(3341, 1)


In [ ]:
# parameters are weight vector and bias
# total params= 4 (3 weights + 1 bias)

In [20]:
def mse(y, y_hat):
    N = y.shape[0]
    loss = (1 / N) * np.sum((y - y_hat) ** 2)
    return loss

loss_value = mse(y_train.reshape(-1,1), y_hat)
print("MSE Loss:", loss_value)

# squaring penalizes large errors more strongly
# large errors become expensive because they are squared

MSE Loss: 141.534046692607


In [21]:
def compute_gradients(X, y, y_hat):
    N = X.shape[0]
    error = y_hat - y   
    
    dw = (2 / N) * (X.T @ error)    
    db = (2 / N) * np.sum(error)     
    return dw, db

In [22]:
def train(X, y, w, b, lr=0.01, epochs=1000):
    
    for epoch in range(epochs):
        y_hat = X @ w + b
        loss = mse(y, y_hat)
        dw, db = compute_gradients(X, y, y_hat)
        w = w - lr * dw
        b = b - lr * db
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss}")
    
    return w, b

In [23]:
y_train = y_train.reshape(-1, 1)
w = np.zeros((X_train.shape[1], 1))
b = 0.0

w, b = train(X_train, y_train, w, b, lr=0.01, epochs=1000)

Epoch 0, Loss: 141.534046692607
Epoch 100, Loss: 9.532421977038066
Epoch 200, Loss: 7.266727848558335
Epoch 300, Loss: 7.219909527350806
Epoch 400, Loss: 7.212628774869934
Epoch 500, Loss: 7.206482792037636
Epoch 600, Loss: 7.200741513708936
Epoch 700, Loss: 7.195350337149574
Epoch 800, Loss: 7.19027556342162
Epoch 900, Loss: 7.1854902432087915


In [ ]:
# gradient tells us the direction in which the loss increases fastest
# subtracting it moves parameters in direction of steepest decrease which reduces the loss

In [26]:
def grad_w(X, y, y_hat):
    N = X.shape[0]
    error = y_hat - y                
    dW = (2 / N) * (X.T @ error)     
    
    return dW
def grad_b(y, y_hat):
    N = y.shape[0]
    error = y_hat - y
    db = (2 / N) * np.sum(error)  
    
    return db
# Forward pass
y_hat = X_train @ w + b

dW = grad_w(X_train, y_train, y_hat)
db = grad_b(y_train, y_hat)

print("w shape :", w.shape)
print("dW shape:", dW.shape)
print("b type :", type(b))
print("db type:", type(db))
# large gradient means the loss is very sensitive to parameter changes
# if learning rate is too large, updates overshoot the minimum

w shape : (3, 1)
dW shape: (3, 1)
b type : <class 'numpy.float64'>
db type: <class 'numpy.float64'>


In [27]:
np.random.seed(42)

d = X_train.shape[1]
w = np.random.randn(d, 1) * 0.01  
b = 0.0

lr = 0.01
epochs = 1000
k = 100   
for epoch in range(epochs):
    y_hat = X_train @ w + b
    loss = mse(y_train, y_hat)
    dW = grad_w(X_train, y_train, y_hat)
    db = grad_b(y_train, y_hat)
    w = w - lr * dW
    b = b - lr * db
    if epoch % k == 0:
        print(f"Epoch {epoch}, Loss: {loss}")
# Initial expectation:
# loss may decrease faster at the beginning and then slow near minimum

# Revised expectation after training:
# Loss decreases quickly in early epochs, then slows as gradients become smaller near optimum.

Epoch 0, Loss: 141.49739178228006
Epoch 100, Loss: 9.532892500026914
Epoch 200, Loss: 7.267146842603629
Epoch 300, Loss: 7.2202880354819605
Epoch 400, Loss: 7.212974775507919
Epoch 500, Loss: 7.2068019873935905
Epoch 600, Loss: 7.201038038925488
Epoch 700, Loss: 7.1956272450106
Epoch 800, Loss: 7.1905351558466135
Epoch 900, Loss: 7.185734298140704


In [ ]:
# Compute test predictions
y_test = y_test.reshape(-1, 1)
y_test_hat = X_test @ w + b
test_mse = mse(y_test, y_test_hat)
print("Test MSE:", test_mse)

def mae(y, y_hat):
    N = y.shape[0]
    return (1 / N) * np.sum(np.abs(y - y_hat))
test_mae = mae(y_test, y_test_hat)
print("Test MAE:", test_mae)

print("\nFive Example Predictions:")
for i in range(5):
    true_age = y_test[i][0]
    pred_age = y_test_hat[i][0]
    abs_error = abs(true_age - pred_age)
    print(f"True age: {true_age:.2f}, "
        f"Predicted age: {pred_age:.2f}, "
        f"Absolute error: {abs_error:.2f}")

# systematic errors:
# the model often underestimates very old abalones and overestimates very young abalones
# this happens because linear regression cannot perfectly model non-linear biological growth patterns

# observed bias:
# predictions tend to move toward the mean age
# this is a typical bias of linear regression

Test MSE: 6.2074384457364244
Test MAE: 1.8675075723672943

Five Example Predictions:
True age: 10.50, Predicted age: 10.03, Absolute error: 0.47
True age: 11.50, Predicted age: 13.54, Absolute error: 2.04
True age: 10.50, Predicted age: 12.45, Absolute error: 1.95
True age: 11.50, Predicted age: 9.20, Absolute error: 2.30
True age: 7.50, Predicted age: 8.34, Absolute error: 0.84
